In [0]:
%pip install databricks-langchain langchain sentencepiece transformers==4.45.2 torch IndicTransToolkit

In [0]:
dbutils.library.restartPython()

In [0]:
# 1. Create a text input widget at the top of your notebook
dbutils.widgets.password("hf_token_widget", "", "Enter HuggingFace Token")

# 2. Pull the token from the widget and login
from huggingface_hub import login

token = dbutils.widgets.get("hf_token_widget")

if token:
    login(token=token)
    #print("✓ Successfully authenticated with HuggingFace")
else:
   # print("⚠ Please enter your token in the text box at the top of the page.")

In [0]:
from databricks_langchain import ChatDatabricks

# Step 1: Initialize the in-built LLM
# Ensure the endpoint name matches what you see in the 'Serving' tab
llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct")

# Generate a response using .invoke()
user_prompt = "Explain BNS Section 318 (Cheating) in one simple sentence."

# .invoke returns a ChatMessage object; .content extracts the raw text
response_object = llm.invoke(user_prompt)
english_text = response_object.content

print(f"--- Original English Response ---\n{english_text}")

In [0]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit import IndicProcessor

# Load the small/efficient model
model_name = "ai4bharat/indictrans2-en-indic-dist-200M"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, trust_remote_code=True)
ip = IndicProcessor(inference=True)

def translate_to_hindi(text):
    # Setup language codes
    src_lang, tgt_lang = "eng_Latn", "hin_Deva"
    
    # Process and Translate
    batch = ip.preprocess_batch([text], src_lang=src_lang, tgt_lang=tgt_lang)
    inputs = tokenizer(batch, src_lang=src_lang, return_tensors="pt")
    
    with torch.no_grad():
        generated_tokens = model.generate(**inputs, num_beams=5, max_length=256)
    
    translations = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    return ip.postprocess_batch(translations, lang=tgt_lang)[0]

# Run the translation on the text we generated in Cell 2
hindi_text = translate_to_hindi(english_text)

print(f"\n--- Hindi Translation ---\n{hindi_text}")

In [0]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit import IndicProcessor

# 1. Setup Model and Tokenizer
# We use the 'distilled' version (200M) for the Free Edition to avoid OOM (Out of Memory) errors
model_name = "ai4bharat/indictrans2-en-indic-dist-200M"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, trust_remote_code=True)

# 2. Initialize the IndicProcessor from the toolkit
# This handles the script unification and language-specific formatting
ip = IndicProcessor(inference=True)

# 3. Define the Translation Function
def translate_to_hindi(text):
    # Language codes: eng_Latn (English), hin_Deva (Hindi)
    src_lang, tgt_lang = "eng_Latn", "hin_Deva"
    
    # Preprocess: Handles the specific formatting IndicTrans2 expects
    batch = ip.preprocess_batch([text], src_lang=src_lang, tgt_lang=tgt_lang)
    
    # Tokenize
    inputs = tokenizer(
        batch, 
        truncation=True, 
        padding="longest", 
        return_tensors="pt"
    )
    
    # Generate Translation
    with torch.no_grad():
        generated_tokens = model.generate(
            **inputs, 
            num_beams=5, 
            max_length=256
        )
    
    # Postprocess: Convert tokens back to readable Hindi text
    translations = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    final_output = ip.postprocess_batch(translations, lang=tgt_lang)
    
    return final_output[0]

# 4. Execute using text from Cell 2
# Assuming 'english_text' was the variable created in Cell 2
try:
    hindi_output = translate_to_hindi(english_text)
    print(f"--- Hindi Output ---\n{hindi_output}")
except NameError:
    print("Error: 'english_text' not found. Please run Cell 2 first!")